# 06 — Ensembles, Model Comparison & Test Evaluation

This notebook evaluates candidate models, builds a simple soft-vote ensemble (when possible), reports comparisons, and evaluates on the test split. Sections are robust to missing artefacts and single-class splits.


## 6.0 — Preamble & Utilities

In [1]:

# =====================================================
# 6.0 — Preamble & Utilities
# =====================================================
print(">>> Section 6.0 — Preamble & Utilities")

import os, json, joblib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
from typing import Any, Dict, List, Tuple

from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    precision_score, recall_score
)

# -- Default env (safe if previous notebooks didn't run)
if 'RANDOM_STATE' not in globals():
    RANDOM_STATE = 42
if 'OUT_ROOT' not in globals():
    OUT_ROOT = Path("out")
else:
    OUT_ROOT = Path(OUT_ROOT)
if 'STAGE_ROOT' not in globals():
    STAGE_ROOT = OUT_ROOT / "stage"

OUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "reports").mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "models").mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "preds").mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "figs").mkdir(parents=True, exist_ok=True)

reports_dir = OUT_ROOT / "reports"
models_dir  = OUT_ROOT / "models"
preds_dir   = OUT_ROOT / "preds"
figs_dir    = OUT_ROOT / "figs"
split_dir   = Path(STAGE_ROOT) / "split_preproc"

# ---------- Array helpers (unwrap saved sparse/objects) ----------
def _load_any(path: Path):
    arr = np.load(path, allow_pickle=True)
    if isinstance(arr, np.ndarray) and arr.dtype == object and arr.size == 1:
        try:
            item = arr.item()
            if sp.issparse(item):
                return item.tocsr()
        except Exception:
            pass
    if sp.issparse(arr):
        return arr.tocsr()
    return arr

def _as_2d(X):
    if sp.issparse(X):
        return X.tocsr()
    if isinstance(X, np.ndarray) and X.ndim == 1:
        return X.reshape(-1, 1)
    return X

def _as_1d_int(y):
    y = np.asarray(y).ravel()
    if y.dtype == object and y.size == 1 and isinstance(y.item(), (np.ndarray, list)):
        y = np.asarray(y.item()).ravel()
    try:
        return y.astype(int)
    except Exception:
        return pd.Series(y).astype(str).str.strip().astype(int).to_numpy()

# ---------- Split loaders ----------
def load_val_test_splits():
    if not split_dir.exists():
        return None

    def pick3(prefix: str):
        # prefer combined
        cands = [
            split_dir / f"{prefix}_all.npy",
            split_dir / f"{prefix}.npy"
        ]
        for c in cands:
            if c.exists():
                return _as_2d(_load_any(c))
        return None

    X_val  = pick3("X_val")
    X_test = pick3("X_test")
    y_val  = _load_any(split_dir / "y_val.npy") if (split_dir / "y_val.npy").exists() else None
    y_test = _load_any(split_dir / "y_test.npy") if (split_dir / "y_test.npy").exists() else None

    if X_val is None or X_test is None or y_val is None or y_test is None:
        return None
    return X_val, X_test, _as_1d_int(y_val), _as_1d_int(y_test)

# ---------- Metrics helpers ----------
def _safe_ap(y_true, p1):
    y_true = np.asarray(y_true)
    u = np.unique(y_true)
    if u.size < 2:
        return 0.0 if (u.size == 1 and u[0] == 0) else 1.0
    return float(average_precision_score(y_true, p1))

def _safe_roc(y_true, p1):
    y_true = np.asarray(y_true)
    if np.unique(y_true).size < 2:
        return 0.5
    return float(roc_auc_score(y_true, p1))

def _get_classes(model):
    # Try top-level
    if hasattr(model, "classes_"):
        return getattr(model, "classes_")
    # Try pipeline final
    try:
        from sklearn.pipeline import Pipeline
        if isinstance(model, Pipeline):
            last = model.steps[-1][1]
            if hasattr(last, "classes_"):
                return last.classes_
    except Exception:
        pass
    return None

def _safe_proba1(model, X):
    # Most sklearn classifiers support predict_proba
    if hasattr(model, "predict_proba"):
        P = model.predict_proba(X)
        if P.ndim == 1:  # unlikely, but guard
            return P
        if P.shape[1] == 2:
            return P[:, 1]
        if P.shape[1] == 1:
            classes = _get_classes(model)
            if classes is not None and len(classes) == 1:
                c = classes[0]
                if c == 0:
                    return np.zeros((P.shape[0],), dtype=float)
                if c == 1:
                    return np.ones((P.shape[0],), dtype=float)
            # Default: assume the single column is the only observed class (often 0); treat as P(y==1)=0
            return np.zeros((P.shape[0],), dtype=float)
    # decision_function fallback
    if hasattr(model, "decision_function"):
        z = model.decision_function(X)
        z = np.asarray(z).ravel()
        # map to (0,1) with logistic
        return 1.0 / (1.0 + np.exp(-z))
    # predict fallback (hard labels)
    yhat = np.asarray(model.predict(X)).ravel()
    return (yhat == 1).astype(float)

def evaluate_model(name, model, X, y):
    p1 = _safe_proba1(model, X)
    pred = (p1 >= 0.5).astype(int)
    row = {
        "name": name,
        "n": int(len(y)),
        "AP": _safe_ap(y, p1),
        "ROC": _safe_roc(y, p1),
        "F1": float(f1_score(y, pred, zero_division=0)),
        "P":  float(precision_score(y, pred, zero_division=0)),
        "R":  float(recall_score(y, pred, zero_division=0)),
    }
    return row

# ---------- Model discovery ----------
def discover_models():
    cands = []
    for p in sorted(models_dir.glob("*.joblib")) + sorted(models_dir.glob("*.pkl")):
        cands.append(p)
    return cands

print(f"[6.0] OUT_ROOT={OUT_ROOT}")
print(f"[6.0] STAGE_ROOT={STAGE_ROOT}")
print(f"[6.0] models_dir contains {len(list(models_dir.glob('*')))} files")


>>> Section 6.0 — Preamble & Utilities


[6.0] OUT_ROOT=out
[6.0] STAGE_ROOT=out/stage
[6.0] models_dir contains 6 files


## 6.1 — Validation Evaluation of Candidate Models

In [2]:

# =====================================================
# 6.1 — Validation Evaluation of Candidate Models
# =====================================================
print(">>> Section 6.1 — Validation Evaluation of Candidate Models")

splits = load_val_test_splits()
if splits is None:
    print("[6.1] No staged VAL/TEST splits found; creating empty evaluation artefacts.")
    results = []
    payload = {"results": results, "candidates": []}
    (reports_dir / "6_1_val_evaluation.csv").write_text("name,n,AP,ROC,F1,P,R\n")
    (reports_dir / "models_evaluated.json").write_text(json.dumps(payload, indent=2))
else:
    X_val, X_test, y_val, y_test = splits
    # report class balance
    print(f"[6.1] VAL classes={np.unique(y_val, return_counts=True)}; TEST classes={np.unique(y_test, return_counts=True)}")

    # Discover & evaluate models
    results = []
    candidates = []
    for path in discover_models():
        name = path.name
        try:
            mdl = joblib.load(path)
            row = evaluate_model(f"VAL::{name}", mdl, X_val, y_val)
            results.append(row)
            candidates.append(str(path))
        except Exception as e:
            print(f"[6.1] Skip {name}: {e}")

    # Persist
    df = pd.DataFrame(results)
    df.to_csv(reports_dir / "6_1_val_evaluation.csv", index=False)
    payload = {"results": results, "candidates": candidates}
    (reports_dir / "models_evaluated.json").write_text(json.dumps(payload, indent=2))
    print(f"[6.1] Wrote VAL eval → {reports_dir / '6_1_val_evaluation.csv'}")
    print(f"[6.1] Catalogued {len(candidates)} models → models_evaluated.json")


>>> Section 6.1 — Validation Evaluation of Candidate Models
[6.1] VAL classes=(array([0, 1]), array([1814,  186])); TEST classes=(array([0, 1]), array([1780,  220]))


[6.1] Skip baseline_model.joblib: X has 500 features, but LogisticRegression is expecting 23 features as input.
[6.1] Skip champion_pipeline.joblib: Can't get attribute 'ensure_min_1_feature' on <module '__main__'>


[6.1] Skip lgbm_bo_payload.pkl: X has 500 features, but LGBMClassifier is expecting 1503 features as input.
[6.1] Wrote VAL eval → out/reports/6_1_val_evaluation.csv
[6.1] Catalogued 1 models → models_evaluated.json


## 6.2 — Simple Blending (Soft Vote)

In [3]:

# =====================================================
# 6.2 — Simple Blending (Soft Vote)
# =====================================================
print(">>> Section 6.2 — Simple Blending (Soft Vote)")

class SoftVoteEnsembler:
    def __init__(self, models):
        self.models = models  # list of (name, model)

    def fit(self, X=None, y=None):
        return self

    def predict_proba(self, X):
        if len(self.models) == 0:
            raise RuntimeError("No base models.")
        P = None
        for _, m in self.models:
            p1 = _safe_proba1(m, X).reshape(-1, 1)
            if P is None:
                P = p1
            else:
                P = P + p1
        P = P / len(self.models)
        # return 2-column proba-like array [P0, P1]
        return np.hstack([1 - P, P])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

# Build ensemble only if we have splits and ≥2 viable models
built = False
try:
    payload = json.loads((reports_dir / "models_evaluated.json").read_text())
    candidates = payload.get("candidates", [])
    if len(candidates) >= 2:
        # Load models
        models = []
        for p in candidates:
            try:
                m = joblib.load(p)
                models.append((Path(p).name, m))
            except Exception as e:
                print(f"[6.2] could not load {p}: {e}")
        if len(models) >= 2:
            ensemble = SoftVoteEnsembler(models)
            joblib.dump(ensemble, models_dir / "ensemble_softvote.joblib")
            print(f"[6.2] Saved ensemble → {models_dir / 'ensemble_softvote.joblib'}")

            # If VAL split is available, evaluate and append to JSON
            splits = load_val_test_splits()
            if splits is not None:
                X_val, X_test, y_val, y_test = splits
                row = evaluate_model("VAL::ensemble_softvote.joblib", ensemble, X_val, y_val)
                payload["results"].append(row)
                (reports_dir / "models_evaluated.json").write_text(json.dumps(payload, indent=2))
            built = True
    else:
        print("[6.2] Not enough candidates to build ensemble (need ≥2).")
except Exception as e:
    print(f"[6.2] Skipped ensemble build: {e}")


>>> Section 6.2 — Simple Blending (Soft Vote)
[6.2] Not enough candidates to build ensemble (need ≥2).


## 6.3 — Reporting & Comparison

In [4]:

# =====================================================
# 6.3 — Reporting & Comparison
# =====================================================
print(">>> Section 6.3 — Reporting & Comparison")

try:
    payload = json.loads((reports_dir / "models_evaluated.json").read_text())
    df = pd.DataFrame(payload.get("results", []))
    if not df.empty:
        # Keep only VAL rows here for comparison
        df_val = df[df["name"].str.startswith("VAL::")].copy()
        df_val.to_csv(reports_dir / "6_3_model_comparison.csv", index=False)
        print(f"[6.3] Wrote comparison → {reports_dir / '6_3_model_comparison.csv'}")
    else:
        (reports_dir / "6_3_model_comparison.csv").write_text("name,n,AP,ROC,F1,P,R\n")
        print("[6.3] No results to compare; wrote empty CSV.")
except Exception as e:
    print(f"[6.3] Skipped comparison: {e}")


>>> Section 6.3 — Reporting & Comparison
[6.3] Wrote comparison → out/reports/6_3_model_comparison.csv


## 6.4 — Test Evaluation & Predictions

In [5]:

# =====================================================
# 6.4 — Test Evaluation & Predictions
# =====================================================
print(">>> Section 6.4 — Test Evaluation & Predictions")

splits = load_val_test_splits()
if splits is None:
    print("[6.4] No VAL/TEST splits found; skipping test evaluation.")
else:
    X_val, X_test, y_val, y_test = splits
    try:
        payload = json.loads((reports_dir / "models_evaluated.json").read_text())
    except Exception:
        payload = {"results": [], "candidates": []}

    # Load every candidate + ensemble if present
    model_paths = [Path(p) for p in payload.get("candidates", []) if Path(p).exists()]
    ens_path = models_dir / "ensemble_softvote.joblib"
    if ens_path.exists():
        model_paths.append(ens_path)

    rows = []
    for path in model_paths:
        try:
            mdl = joblib.load(path)
            row = evaluate_model(f"TEST::{path.name}", mdl, X_test, y_test)
            rows.append(row)
            # write predictions
            p1 = _safe_proba1(mdl, X_test)
            pred = (p1 >= 0.5).astype(int)
            out_csv = preds_dir / f"{path.stem}_test_predictions.csv"
            pd.DataFrame({"proba1": p1, "pred": pred}).to_csv(out_csv, index=False)
        except Exception as e:
            print(f"[6.4] Skip {path.name}: {e}")

    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(reports_dir / "6_4_test_evaluation.csv", index=False)
        print(f"[6.4] Wrote test eval → {reports_dir / '6_4_test_evaluation.csv'}")
    else:
        print("[6.4] No models could be evaluated on TEST.")


>>> Section 6.4 — Test Evaluation & Predictions
[6.4] Wrote test eval → out/reports/6_4_test_evaluation.csv


In [6]:
# --- Persist best tree metric for downstream notebooks ---
from pathlib import Path
import json

metrics_dir = Path("out") / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)

# Replace these with your real best scores at the end of 04/06
best_tree_val_acc = float(globals().get("BEST_TREE_VAL_ACC", 0.0))
best_tree_name    = str(globals().get("BEST_TREE_NAME", "unknown"))

payload = {"best_tree_name": best_tree_name, "best_tree_val_acc": best_tree_val_acc}
with open(metrics_dir / "best_tree.json", "w") as f:
    json.dump(payload, f)

print(f"[persist] Wrote best tree → {metrics_dir/'best_tree.json'}: {payload}")


[persist] Wrote best tree → out/metrics/best_tree.json: {'best_tree_name': 'unknown', 'best_tree_val_acc': 0.0}
